# Lesson 12B: Variational Autoencoders - Practical Implementation

## Introduction

**Reference**: Kingma, D. P., & Welling, M. (2014). "Auto-Encoding Variational Bayes". *International Conference on Learning Representations (ICLR)*.

The Variational Autoencoder (VAE) is a generative model that learns a probabilistic mapping between data and a latent representation. Unlike standard autoencoders, VAEs learn a structured latent space by enforcing a prior distribution, enabling principled generation of new samples.

**Objective**: Implement and train a VAE on the handwritten digits dataset, demonstrating:
1. Probabilistic encoding with the reparameterization trick
2. Evidence Lower Bound (ELBO) optimization
3. Latent space exploration and generation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits

torch.manual_seed(42)
np.random.seed(42)
print("Libraries loaded successfully")

## Theoretical Background

### Probabilistic Formulation

Given observed data x, we wish to model the generative process:

1. Sample latent code: z ~ p(z) = N(0, I)
2. Generate data: x ~ p_θ(x|z)

The true posterior p(z|x) is intractable, so we approximate it with q_φ(z|x).

### Evidence Lower Bound (ELBO)

The log-likelihood can be decomposed as:

$$\log p_\theta(x) = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x) \| p(z)) + D_{KL}(q_\phi(z|x) \| p_\theta(z|x))$$

Since KL divergence is non-negative, we maximize the ELBO:

$$\mathcal{L}(\theta, \phi; x) = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x) \| p(z))$$

### Component Analysis

**Reconstruction Term**: $\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]$
- Ensures decoder reconstructs input from encoded representation
- Typically implemented as MSE or binary cross-entropy

**KL Regularization**: $D_{KL}(q_\phi(z|x) \| p(z))$
- Regularizes latent space to match prior p(z) = N(0, I)
- For Gaussian encoder q_φ(z|x) = N(μ, σ²I):

$$D_{KL}(q_\phi(z|x) \| p(z)) = -\frac{1}{2}\sum_{j=1}^J (1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2)$$

where J is the latent dimension.

### Reparameterization Trick

To enable backpropagation through sampling, we reparameterize:

$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

This separates the stochastic component (ε) from the learnable parameters (μ, σ).

In [ ]:
class VAE(nn.Module):
    """
    Variational Autoencoder implementation
    
    Architecture:
        Encoder: x → h → (μ, log σ²)
        Latent: z = μ + σ * ε
        Decoder: z → h → x̂
    """
    def __init__(self, input_dim=64, hidden_dim=32, latent_dim=8):
        super().__init__()
        
        # Encoder: q_φ(z|x)
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder: p_θ(x|z)
        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)
        
        self.latent_dim = latent_dim
    
    def encode(self, x):
        """
        Encoder: q_φ(z|x) = N(μ(x), σ²(x))
        Returns: (μ, log σ²)
        """
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick: z = μ + σ * ε
        where ε ~ N(0, I)
        
        Args:
            mu: Mean of q(z|x)
            logvar: Log variance of q(z|x)
        Returns:
            z: Latent sample
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps
    
    def decode(self, z):
        """
        Decoder: p_θ(x|z)
        Returns: Reconstructed x
        """
        h = F.relu(self.fc2(z))
        return torch.sigmoid(self.fc3(h))
    
    def forward(self, x):
        """
        Forward pass through VAE
        Returns: (reconstruction, μ, log σ²)
        """
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

vae = VAE(input_dim=64, hidden_dim=32, latent_dim=8)
print(f"VAE architecture initialized")
print(f"Total parameters: {sum(p.numel() for p in vae.parameters()):,}")

## Dataset Preparation

In [ ]:
# Load MNIST-like digits dataset
digits = load_digits()
X = digits.data / 16.0  # Normalize to [0, 1]
y = digits.target

print(f"Dataset shape: {X.shape}")
print(f"Number of samples: {len(X)}")
print(f"Feature dimension: {X.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")

# Convert to PyTorch tensor
X_tensor = torch.FloatTensor(X)

# Visualize samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Digit: {y[i]}')
    ax.axis('off')
plt.suptitle('Sample Handwritten Digits', fontweight='bold')
plt.tight_layout()
plt.show()
print("Data loaded and visualized")

## Loss Function Implementation

The VAE loss combines reconstruction and KL divergence:

$$\mathcal{L} = \mathcal{L}_{\text{recon}} + \beta \cdot \mathcal{L}_{\text{KL}}$$

where β = 1 for standard VAE (β-VAE uses β ≠ 1 for disentanglement).

In [ ]:
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    """
    Compute VAE loss (negative ELBO)
    
    Args:
        recon_x: Reconstructed data
        x: Original data
        mu: Latent mean
        logvar: Latent log variance
        beta: Weight for KL term (β-VAE parameter)
    
    Returns:
        total_loss, reconstruction_loss, kl_divergence
    """
    # Reconstruction loss: E[log p(x|z)]
    # Using MSE as proxy for Gaussian likelihood
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    
    # KL divergence: KL(q(z|x) || p(z))
    # Closed form for diagonal Gaussian:
    # -0.5 * Σ(1 + log(σ²) - μ² - σ²)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    # Total loss (negative ELBO)
    total_loss = recon_loss + beta * kld
    
    return total_loss, recon_loss, kld

# Validate loss computation
recon, mu, logvar = vae(X_tensor[:10])
total, recon_l, kld_l = vae_loss(recon, X_tensor[:10], mu, logvar)
print(f"Loss components (untrained model):")
print(f"  Reconstruction: {recon_l.item():.2f}")
print(f"  KL Divergence: {kld_l.item():.2f}")
print(f"  Total: {total.item():.2f}")

## Training Procedure

We optimize the ELBO using stochastic gradient descent.

In [ ]:
# Training configuration
optimizer = optim.Adam(vae.parameters(), lr=1e-3)
n_epochs = 100
beta = 1.0  # Standard VAE (set β > 1 for β-VAE)

# Loss tracking
losses = {'total': [], 'recon': [], 'kld': []}

print(f"Training VAE for {n_epochs} epochs...")
vae.train()

for epoch in range(n_epochs):
    # Forward pass
    recon, mu, logvar = vae(X_tensor)
    total_loss, recon_loss, kld_loss = vae_loss(recon, X_tensor, mu, logvar, beta)
    
    # Backward pass
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    
    # Logging
    losses['total'].append(total_loss.item() / len(X_tensor))
    losses['recon'].append(recon_loss.item() / len(X_tensor))
    losses['kld'].append(kld_loss.item() / len(X_tensor))
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}")
        print(f"  Total Loss: {losses['total'][-1]:.3f}")
        print(f"  Reconstruction: {losses['recon'][-1]:.3f}")
        print(f"  KL Divergence: {losses['kld'][-1]:.3f}")

print("Training completed")

## Training Analysis

In [ ]:
# Visualize training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax = axes[0]
ax.plot(losses['total'], label='Total Loss (Negative ELBO)', linewidth=2)
ax.plot(losses['recon'], label='Reconstruction Loss', linewidth=2, alpha=0.7)
ax.plot(losses['kld'], label='KL Divergence', linewidth=2, alpha=0.7)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (per sample)', fontsize=12)
ax.set_title('VAE Training Dynamics', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# Reconstruction quality scatter
vae.eval()
with torch.no_grad():
    recon, _, _ = vae(X_tensor[:100])
    recon = recon.numpy()

ax = axes[1]
ax.scatter(X[:100].flatten(), recon.flatten(), alpha=0.3, s=10)
ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect reconstruction')
ax.set_xlabel('Original pixel value', fontsize=12)
ax.set_ylabel('Reconstructed pixel value', fontsize=12)
ax.set_title('Reconstruction Quality', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Training analysis complete")

## Reconstruction Evaluation

In [ ]:
# Visual comparison of reconstructions
fig, axes = plt.subplots(4, 10, figsize=(15, 6))

with torch.no_grad():
    recon, _, _ = vae(X_tensor[:10])
    recon = recon.numpy()

for i in range(10):
    # Original
    axes[0, i].imshow(X[i].reshape(8, 8), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', fontsize=11, rotation=0, ha='right', va='center')
    
    # Reconstructed
    axes[1, i].imshow(recon[i].reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Reconstruction', fontsize=11, rotation=0, ha='right', va='center')
    
    # Absolute difference
    diff = np.abs(X[i] - recon[i]).reshape(8, 8)
    axes[2, i].imshow(diff, cmap='hot')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('|Error|', fontsize=11, rotation=0, ha='right', va='center')
    
    # True label
    axes[3, i].text(0.5, 0.5, str(y[i]), fontsize=16, ha='center', va='center')
    axes[3, i].axis('off')
    if i == 0:
        axes[3, i].set_ylabel('Label', fontsize=11, rotation=0, ha='right', va='center')

plt.suptitle('VAE Reconstruction Analysis', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

# Quantitative evaluation
with torch.no_grad():
    recon_full, _, _ = vae(X_tensor)
    mse = F.mse_loss(recon_full, X_tensor).item()
print(f"Mean reconstruction MSE: {mse:.6f}")

## Latent Space Sampling

We generate new samples by sampling from the prior p(z) = N(0, I) and decoding.

In [ ]:
# Sample from latent space
n_samples = 10
z_samples = torch.randn(n_samples, 8)  # Sample from N(0, I)

with torch.no_grad():
    generated = vae.decode(z_samples).numpy()

# Visualize generated samples
fig, axes = plt.subplots(1, n_samples, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(generated[i].reshape(8, 8), cmap='gray')
    ax.axis('off')
plt.suptitle('Generated Samples from Prior p(z) = N(0, I)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()
print("Generated new digit samples from latent space")

## Discussion

### Key Results

1. **Smooth Latent Space**: The KL regularization enforces a structured latent space following N(0, I)
2. **Generative Capability**: Sampling from the prior generates plausible new digits
3. **Reconstruction-Regularization Tradeoff**: The β parameter controls this balance

### Theoretical Insights

**Why VAE works**:
- ELBO maximization approximates maximum likelihood
- KL regularization prevents "holes" in latent space
- Reparameterization enables end-to-end training

**Comparison with Standard Autoencoders**:
- VAE: Probabilistic, regularized latent space, generative
- AE: Deterministic, unregularized, primarily for reconstruction

### Extensions

1. **β-VAE** (Higgins et al., 2017): Use β > 1 for disentangled representations
2. **Conditional VAE** (Sohn et al., 2015): Condition on class labels
3. **Hierarchical VAE** (Sønderby et al., 2016): Multiple latent layers
4. **VQ-VAE** (van den Oord et al., 2017): Discrete latent variables

## References

1. Kingma, D. P., & Welling, M. (2014). "Auto-Encoding Variational Bayes". *International Conference on Learning Representations (ICLR)*.

2. Rezende, D. J., Mohamed, S., & Wierstra, D. (2014). "Stochastic Backpropagation and Approximate Inference in Deep Generative Models". *International Conference on Machine Learning (ICML)*, 1278-1286.

3. Higgins, I., Matthey, L., Pal, A., et al. (2017). "β-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework". *International Conference on Learning Representations (ICLR)*.

4. Doersch, C. (2016). "Tutorial on Variational Autoencoders". *arXiv preprint arXiv:1606.05908*.

5. Kingma, D. P., & Welling, M. (2019). "An Introduction to Variational Autoencoders". *Foundations and Trends in Machine Learning*, 12(4), 307-392.